In [6]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [7]:
price = pd.read_csv("../data/forex_price_features.csv")

price["DateTime"] = pd.to_datetime(price["DateTime"])

In [8]:
price = price.dropna().copy()

In [9]:
price["Target"] = np.where(
    price["Close"].shift(-1) > price["Close"],
    1,
    0
)

price = price.dropna().copy()

In [10]:
features = [
    "Open",
    "High",
    "Low",
    "Close",
    "Volume",
    "Spread",
    "MA_5",
    "MA_20",
    "RSI",
    "MACD",
    "MACD_Signal",
    "BB_Upper",
    "BB_Middle",
    "BB_Lower",
    "Price_Change",
    "High_Low_Range",
    "Return"
]

print(price.columns.tolist())
X = price[features]
y = price["Target"]

['DateTime', 'PairID', 'PairName', 'Open', 'High', 'Low', 'Close', 'Volume', 'Spread', 'MA_5', 'MA_20', 'Price_Change', 'High_Low_Range', 'Return', 'RSI', 'MACD', 'MACD_Signal', 'MACD_Histogram', 'BB_Middle', 'BB_Upper', 'BB_Lower', 'Target']


In [11]:
print(price.columns.tolist())

['DateTime', 'PairID', 'PairName', 'Open', 'High', 'Low', 'Close', 'Volume', 'Spread', 'MA_5', 'MA_20', 'Price_Change', 'High_Low_Range', 'Return', 'RSI', 'MACD', 'MACD_Signal', 'MACD_Histogram', 'BB_Middle', 'BB_Upper', 'BB_Lower', 'Target']


In [12]:
from ta.momentum import RSIIndicator
from ta.trend import MACD
from ta.volatility import BollingerBands

In [13]:
# Moving Averages
price["MA_5"] = price["Close"].rolling(window=5).mean()
price["MA_20"] = price["Close"].rolling(window=20).mean()

# Basic Features
price["Price_Change"] = price["Close"] - price["Open"]
price["High_Low_Range"] = price["High"] - price["Low"]
price["Return"] = price["Close"].pct_change()

# RSI
price["RSI"] = RSIIndicator(
    close=price["Close"],
    window=14
).rsi()

# MACD
macd = MACD(close=price["Close"])

price["MACD"] = macd.macd()
price["MACD_Signal"] = macd.macd_signal()

# Bollinger Bands
bb = BollingerBands(
    close=price["Close"],
    window=20,
    window_dev=2
)

price["BB_Upper"] = bb.bollinger_hband()
price["BB_Middle"] = bb.bollinger_mavg()
price["BB_Lower"] = bb.bollinger_lband()

# Remove rows with missing values
price = price.dropna().copy()

In [14]:
print(price.columns.tolist())

['DateTime', 'PairID', 'PairName', 'Open', 'High', 'Low', 'Close', 'Volume', 'Spread', 'MA_5', 'MA_20', 'Price_Change', 'High_Low_Range', 'Return', 'RSI', 'MACD', 'MACD_Signal', 'MACD_Histogram', 'BB_Middle', 'BB_Upper', 'BB_Lower', 'Target']


In [15]:
X = price[features]
y = price["Target"]

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Shape:", X_train.shape)
print("Testing Shape :", X_test.shape)

Training Shape: (347, 17)
Testing Shape : (87, 17)


In [17]:
log_model = LogisticRegression(max_iter=1000)

log_model.fit(X_train, y_train)

print("✅ Logistic Regression Model Trained")

✅ Logistic Regression Model Trained


In [18]:
log_pred = log_model.predict(X_test)

log_accuracy = accuracy_score(y_test, log_pred)

print(f"Logistic Regression Accuracy: {log_accuracy * 100:.2f}%")

Logistic Regression Accuracy: 49.43%


In [19]:
print(classification_report(y_test, log_pred))

              precision    recall  f1-score   support

           0       0.52      0.48      0.50        46
           1       0.47      0.51      0.49        41

    accuracy                           0.49        87
   macro avg       0.50      0.50      0.49        87
weighted avg       0.50      0.49      0.49        87



In [20]:
from xgboost import XGBClassifier

In [21]:
# Create XGBoost Model

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    eval_metric="logloss"
)

# Train Model

xgb_model.fit(X_train, y_train)

print("✅ XGBoost Model Trained Successfully")

✅ XGBoost Model Trained Successfully


In [22]:
# Predict on Test Data

xgb_pred = xgb_model.predict(X_test)

xgb_accuracy = accuracy_score(y_test, xgb_pred)

print(f"XGBoost Accuracy: {xgb_accuracy * 100:.2f}%")

XGBoost Accuracy: 43.68%


In [23]:
print(classification_report(y_test, xgb_pred))

              precision    recall  f1-score   support

           0       0.47      0.46      0.46        46
           1       0.40      0.41      0.41        41

    accuracy                           0.44        87
   macro avg       0.44      0.44      0.44        87
weighted avg       0.44      0.44      0.44        87



In [24]:
# Train Random Forest

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

# Predictions

rf_pred = rf_model.predict(X_test)

# Accuracy

rf_accuracy = accuracy_score(y_test, rf_pred)

print(f"Random Forest Accuracy: {rf_accuracy * 100:.2f}%")

Random Forest Accuracy: 55.17%


In [25]:
comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "XGBoost"
    ],
    "Accuracy": [
        log_accuracy,
        rf_accuracy,
        xgb_accuracy
    ]
})

comparison["Accuracy"] = comparison["Accuracy"] * 100

comparison = comparison.sort_values(
    by="Accuracy",
    ascending=False
)

print(comparison)

                 Model   Accuracy
1        Random Forest  55.172414
0  Logistic Regression  49.425287
2              XGBoost  43.678161
